# P0-6 · PyTorch cơ bản: tensor, autograd, nn.Module

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import torch
print(torch.__version__)

2.15.0+cu132


In [ ]:
import torch
x = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])        # từ list, giống np.array
print(x)
print(x.shape, x.dtype, x.ndim)         # torch.Size([2, 3]) torch.float32 2

tensor([[1., 2., 3.],
        [4., 5., 6.]])
torch.Size([2, 3]) torch.float32 2


In [ ]:
torch.manual_seed(0)                    # cố định số ngẫu nhiên để chạy lại ra giống nhau
print(torch.zeros(2, 3))                # toàn 0, shape (2, 3)
print(torch.arange(0, 10, 2))           # 0 2 4 6 8, giống np.arange
print(torch.randn(2, 3))                # số ngẫu nhiên phân phối chuẩn

tensor([[0., 0., 0.],
        [0., 0., 0.]])
tensor([0, 2, 4, 6, 8])
tensor([[ 1.5410, -0.2934, -2.1788],
        [ 0.5684, -1.0845, -1.3986]])


In [ ]:
print(torch.tensor([1, 2, 3]).dtype)    # số nguyên → int64
print(torch.tensor([1., 2.]).dtype)     # số thực → float32 (numpy mặc định float64)

torch.int64
torch.float32


In [ ]:
import numpy as np
a = np.array([1.5, 2.5, 3.5])
t = torch.from_numpy(a)                 # numpy → tensor (dùng chung bộ nhớ với a)
print(t, t.dtype)                       # dtype vẫn là float64 như numpy
print(t.float().dtype)                  # đổi sang float32
print(t.numpy())                        # tensor → numpy
print(t.sum().item())                   # tensor 1 phần tử → số Python thường

tensor([1.5000, 2.5000, 3.5000], dtype=torch.float64) torch.float64
torch.float32
[1.5 2.5 3.5]
7.5


In [ ]:
x = torch.arange(6.).reshape(2, 3)
print(x.sum(dim=0))                     # cộng dọc 2 hàng → 3 số
print(x.mean(dim=1))                    # trung bình mỗi hàng → 2 số
print(x @ x.T)                          # (2, 3) @ (3, 2) → (2, 2)
print(x.argmax(dim=1))                  # vị trí lớn nhất mỗi hàng

tensor([3., 5., 7.])
tensor([1., 4.])
tensor([[ 5., 14.],
        [14., 50.]])
tensor([2, 2])


In [ ]:
img = torch.rand(32, 32, 3)             # ảnh 32x32, 3 kênh màu, kiểu "cao, rộng, kênh"
chw = img.permute(2, 0, 1)              # đổi thành (3, 32, 32)
batch = chw.unsqueeze(0)                # thêm trục batch: (1, 3, 32, 32)
print(chw.shape, batch.shape)

torch.Size([3, 32, 32]) torch.Size([1, 3, 32, 32])


In [ ]:
if torch.cuda.is_available():
    device = "cuda"                     # GPU NVIDIA (Colab: Runtime → Change runtime type → T4 GPU)
elif torch.backends.mps.is_available():
    device = "mps"                      # GPU của máy Mac chip Apple
else:
    device = "cpu"
x = torch.randn(3, 3).to(device)
print(device, x.device)

cpu cpu


In [ ]:
w = torch.tensor(2.0, requires_grad=True)
y = w ** 2 + 3 * w                      # y = w² + 3w
y.backward()                            # tính dy/dw
print(y.item(), w.grad.item())          # y = 10; dy/dw = 2w + 3 = 7 tại w = 2

10.0 7.0


In [ ]:
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
x, y = torch.tensor(2.0), torch.tensor(5.0)

pred = w * x + b                        # 2
loss = (pred - y) ** 2                  # (2 - 5)² = 9
loss.backward()
print(loss.item(), w.grad.item(), b.grad.item())

9.0 -12.0 -6.0


In [ ]:
w = torch.tensor(2.0, requires_grad=True)
for i in range(3):
    y = w ** 2 + 3 * w
    y.backward()
    print("lần", i + 1, "w.grad =", w.grad.item())   # 7, 14, 21: cộng dồn chứ không ghi đè!
# xóa gradient về 0
w.grad.zero_()
print("sau zero_:", w.grad.item())

lần 1 w.grad = 7.0
lần 2 w.grad = 14.0
lần 3 w.grad = 21.0
sau zero_: 0.0


In [ ]:
torch.manual_seed(0)
X = torch.rand(100) * 10                       # 100 giá trị x từ 0 tới 10
Y = 2 * X + 1 + torch.randn(100) * 0.5         # đáp án thật: w = 2, b = 1, cộng nhiễu

w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
lr = 0.01                                      # learning rate: độ dài mỗi bước

for step in range(1001):
    pred = w * X + b
    loss = ((pred - Y) ** 2).mean()            # MSE: trung bình bình phương sai số
    loss.backward()                            # tính w.grad, b.grad
    with torch.no_grad():                      # cập nhật tham số, không ghi đồ thị
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad.zero_(); b.grad.zero_()             # xóa gradient cho bước sau
    if step % 200 == 0:
        print(f"bước {step:4d}: loss = {loss.item():.3f}, w = {w.item():.3f}, b = {b.item():.3f}")

bước    0: loss = 149.308, w = 1.389, b = 0.215
bước  200: loss = 0.222, w = 2.011, b = 0.867
bước  400: loss = 0.201, w = 1.982, b = 1.060
bước  600: loss = 0.198, w = 1.971, b = 1.129
bước  800: loss = 0.198, w = 1.968, b = 1.153
bước 1000: loss = 0.198, w = 1.966, b = 1.161


In [ ]:
import torch.nn as nn
lin = nn.Linear(3, 2)                          # 3 đầu vào, 2 đầu ra
print(lin.weight.shape, lin.bias.shape)        # W: (2, 3), b: (2,)
n_params = sum(p.numel() for p in lin.parameters())
print("số tham số:", n_params)                 # 3*2 + 2 = 8
print(lin(torch.randn(4, 3)).shape)            # batch 4 mẫu → (4, 2)

torch.Size([2, 3]) torch.Size([2])
số tham số: 8
torch.Size([4, 2])


In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()                     # dòng bắt buộc đầu tiên
        self.l1 = nn.Linear(2, 16)
        self.l2 = nn.Linear(16, 2)
    def forward(self, x):
        h = torch.relu(self.l1(x))             # lớp 1 rồi ReLU (giữ số dương, số âm thành 0)
        return self.l2(h)                      # lớp 2: 2 điểm số, một cho mỗi lớp

model = MLP()
print(model)
print("số tham số:", sum(p.numel() for p in model.parameters()))   # (2*16+16) + (16*2+2) = 82

MLP(
  (l1): Linear(in_features=2, out_features=16, bias=True)
  (l2): Linear(in_features=16, out_features=2, bias=True)
)
số tham số: 82


In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)    # model MLP ở trên
xb = torch.randn(8, 2)                                      # một batch giả: 8 mẫu, mỗi mẫu 2 số
yb = torch.randint(0, 2, (8,))                              # nhãn 0 hoặc 1

for step in range(3):
    optimizer.zero_grad()          # 1. xóa gradient cũ
    pred = model(xb)               # 2. forward: dự đoán
    loss = loss_fn(pred, yb)       # 3. tính loss
    loss.backward()                # 4. backward: tính gradient
    optimizer.step()               # 5. cập nhật tham số theo gradient
    print(step, loss.item())       # loss giảm dần trên batch này

0 0.7276450395584106
1 0.6813852787017822
2 0.647287905216217


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

X = torch.randn(100, 2)
y = torch.randint(0, 2, (100,))                # nhãn 0 hoặc 1
ds = TensorDataset(X, y)                       # ds[i] trả về cặp (X[i], y[i])
loader = DataLoader(ds, batch_size=32, shuffle=True)
print(len(ds), len(loader))                    # 100 mẫu, 4 batch (32 + 32 + 32 + 4)
for xb, yb in loader:
    print(xb.shape, yb.shape)

100 4
torch.Size([32, 2]) torch.Size([32])
torch.Size([32, 2]) torch.Size([32])
torch.Size([32, 2]) torch.Size([32])
torch.Size([4, 2]) torch.Size([4])


In [ ]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0)
n = 400
X = torch.randn(n, 2)
y = ((X[:, 0] ** 2 + X[:, 1] ** 2) > 1.2).long()   # lớp 1 nếu điểm nằm ngoài vòng tròn
train_ds = TensorDataset(X[:300], y[:300])
val_X, val_y = X[300:], y[300:]                     # 100 điểm giữ riêng để kiểm tra

model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loader = DataLoader(train_ds, batch_size=32, shuffle=True)

for epoch in range(1, 61):
    model.train()
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
    if epoch in (1, 3, 10, 30, 60):
        model.eval()
        with torch.no_grad():
            acc = (model(val_X).argmax(dim=1) == val_y).float().mean()
        print(f"epoch {epoch}: loss batch cuối = {loss.item():.3f}, val accuracy = {acc.item():.2f}")

epoch 1: loss batch cuối = 0.606, val accuracy = 0.61
epoch 3: loss batch cuối = 0.531, val accuracy = 0.85
epoch 10: loss batch cuối = 0.220, val accuracy = 0.98
epoch 30: loss batch cuối = 0.066, val accuracy = 0.98
epoch 60: loss batch cuối = 0.059, val accuracy = 0.98


In [ ]:
torch.save(model.state_dict(), "model.pt")             # lưu ra file

model2 = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))   # cùng kiến trúc
model2.load_state_dict(torch.load("model.pt"))
model2.eval()
with torch.no_grad():
    print(torch.equal(model(val_X), model2(val_X)))   # True: dự đoán giống hệt
print(list(model.state_dict().keys()))

True
['0.weight', '0.bias', '2.weight', '2.bias']


## Bài tập 2: Một bước gradient descent bằng tay {tính tay}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch
w = torch.tensor(1.0, requires_grad=True)
loss = (w * 2 - 5) ** 2
loss.backward()
print(loss.item(), w.grad.item())
with torch.no_grad():
    w -= 0.1 * w.grad
print(w.item(), ((w * 2 - 5) ** 2).item())

9.0 -12.0
2.200000047683716 0.3599998950958252


## Bài tập 4: Sửa ba lỗi {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np, torch, torch.nn as nn
rng = np.random.default_rng(0)
X = rng.normal(size=(200, 3))                       # dữ liệu từ numpy
y = (X[:, 0] + X[:, 1] > 0).astype(float)           # nhãn 0/1

model = nn.Linear(3, 2)
opt = torch.optim.SGD(model.parameters(), lr=0.5)
Xt, yt = torch.from_numpy(X), torch.from_numpy(y)
for step in range(5):
    loss = nn.CrossEntropyLoss()(model(Xt), yt)
    loss.backward()
    opt.step()
    print(step, round(loss.item(), 3))

In [ ]:
import numpy as np, torch, torch.nn as nn
rng = np.random.default_rng(0)
X = rng.normal(size=(200, 3))
y = (X[:, 0] + X[:, 1] > 0).astype(float)

model = nn.Linear(3, 2)
opt = torch.optim.SGD(model.parameters(), lr=0.5)
Xt = torch.from_numpy(X).float()                    # sửa 1: float64 → float32
yt = torch.from_numpy(y).long()                     # sửa 2: nhãn phải là số nguyên int64
for step in range(5):
    opt.zero_grad()                                 # sửa 3: xóa gradient cũ mỗi bước
    loss = nn.CrossEntropyLoss()(model(Xt), yt)
    loss.backward()
    opt.step()
    print(step, round(loss.item(), 3))

0 0.975
1 0.734
2 0.571
3 0.469
4 0.406


## Bài tập 5: Từ numpy sang PyTorch {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np, torch

def softmax_np(Z):
    E = np.exp(Z - Z.max(axis=1, keepdims=True))
    return E / E.sum(axis=1, keepdims=True)

def softmax_t(Z):
    E = torch.exp(Z - Z.max(dim=1, keepdim=True).values)
    return E / E.sum(dim=1, keepdim=True)

def accuracy_t(logits, y):
    return (logits.argmax(dim=1) == y).float().mean().item()

torch.manual_seed(0)
Z = torch.randn(5, 3) * 10
print(np.allclose(softmax_t(Z).numpy(), softmax_np(Z.numpy()), atol=1e-6))
print(torch.allclose(softmax_t(Z), torch.softmax(Z, dim=1)))
print(accuracy_t(Z, torch.tensor([0, 1, 2, 0, 1])))

True
True
0.4000000059604645
